# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to use the `mlcroissant` library to explore and process a FAIR^2-compliant dataset described by a Croissant schema. All data elements are referenced by their `@id` fields for full reproducibility and clarity.

### Dataset Source
The dataset metadata is provided at the following Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load dataset metadata and browse available record sets and fields using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant metadata schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Dataset ID: {metadata.id}")
print(f"Version: {metadata.version}")
print(f"License: {metadata.license}")

## 2. Data Overview
List the available record sets, fields, and their `@id`s. All operations reference entities by their unique `@id`. This ensures reproducibility and transparency for downstream processing.

In [ ]:
# For most Croissant datasets, record sets contain tabular data (e.g., protein abundance).

print("Available record sets in this dataset (showing @id and name):")
record_set_ids = []
for record_set in metadata.record_sets:
    print(f"  @id: {record_set.id}, name: {record_set.name}")
    record_set_ids.append(record_set.id)

# For each record set, list available field @ids and names
for record_set in metadata.record_sets:
    print(f"\nFields for record set '{record_set.name}' (@id: {record_set.id}):")
    for field in record_set.fields:
        desc = field.description if hasattr(field, "description") else ""
        print(f"  Field @id: {field.id:40s} | name: {field.name:30s} | type: {str(field.data_type):12s} {desc}")

## 3. Data Extraction
Extract records from a target record set into a DataFrame for further analysis. Reference all fields and columns by `@id` wherever possible.

In this dataset, the main protein records are usually found in the large tabular record set. Pick the most relevant record set based on the data overview.

In [ ]:
# Use the main tabular record set, which is typically the first one

# If multiple record sets are available, you can set this to your preferred one
main_record_set_id = record_set_ids[0] if record_set_ids else None
assert main_record_set_id is not None, "No record sets found in the Croissant schema!"

print(f"Extracting all records from record set @id: {main_record_set_id}\n")
records_iter = dataset.records(record_set=main_record_set_id)
records = list(records_iter)
df = pd.DataFrame(records)
print(f"Loaded DataFrame for record set {main_record_set_id} with shape:", df.shape)
print("Available columns (these correspond to field @id):\n", list(df.columns))
df.head()

## 4. Exploratory Data Analysis (EDA)
Now let's filter and analyze numeric fields, and group or normalize values as needed. All operations use `@id` references for fields.

In [ ]:
# Identify a numeric field to analyze (based on field listing above)
# Example: Let's find the first field whose dtype is float or int

numeric_field_id = None
for record_set in metadata.record_sets:
    if record_set.id == main_record_set_id:
        for field in record_set.fields:
            dt = str(field.data_type).lower()
            if dt.startswith('float') or dt.startswith('int') or dt == 'number':
                numeric_field_id = field.id
                print(f"Selected numeric field: {numeric_field_id} ({field.name})")
                break
        break
assert numeric_field_id is not None, "No numeric field identified in record set!"

# Convert the chosen field to numeric if needed
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

# Filter: e.g., look at proteins with abundance > threshold
threshold = df[numeric_field_id].mean()
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered records for {numeric_field_id} > mean value ({threshold:.4f}), count: {filtered_df.shape[0]}")

# Normalize the numeric column
filtered_df[f"{numeric_field_id}_normalized"] = (
    (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
)

print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head(10))

# Try to group by a categorical/text field: pick one (e.g., first text field)
group_field_id = None
for record_set in metadata.record_sets:
    if record_set.id == main_record_set_id:
        for field in record_set.fields:
            dt = str(field.data_type).lower()
            if ('text' in dt or 'str' in dt or dt == 'string') and field.id != numeric_field_id:
                group_field_id = field.id
                print(f"Grouping by field: {group_field_id} ({field.name})")
                break
        break
if group_field_id and group_field_id in filtered_df.columns:
    grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().sort_values(ascending=False)
    print(f"Mean values of {numeric_field_id} grouped by {group_field_id}:")
    print(grouped.head())

## 5. Visualization
Let's visualize the distribution of the analyzed numeric field and group comparisons if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Boxplot of the numeric field
plt.figure(figsize=(8,5))
sns.boxplot(data=df, x=numeric_field_id)
plt.title(f"Boxplot of {numeric_field_id}")
plt.show()

# If grouping field is present, show top group means
if group_field_id and group_field_id in filtered_df.columns:
    top_groups = filtered_df.groupby(group_field_id)[numeric_field_id].mean().nlargest(10)
    plt.figure(figsize=(10,6))
    sns.barplot(y=top_groups.index, x=top_groups.values, orient='h')
    plt.xlabel(numeric_field_id)
    plt.ylabel(group_field_id)
    plt.title(f"Top 10 {group_field_id} by mean {numeric_field_id}")
    plt.show()

## 6. Conclusion
- Successfully explored a FAIR^2-compliant mass spectrometry dataset with `mlcroissant`, referencing each entity by its `@id`.
- Loaded record-level data, filtered on quantitative fields, and visualized data distributions and groupings for biological insight.
- By using `@id` fields, the workflow is schema-driven and robust to upstream changes in dataset structure.

For more advanced usage, refer to the [mlcroissant documentation](https://mlcommons.github.io/croissant/).